Step 1: Dataset Collection & Preprocessing

We will use a public domain text dataset (a classic book from Project Gutenberg) for simplicity.



In [1]:
!pip install tensorflow requests

  Using cached requests-2.33.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached absl_py-2.4.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-1-py2.py3-none-macosx_11_0_arm64.whl.metadata (5.2 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached termcolor-3.3.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached wrapt-2.1.2-cp311-cp311-macosx_11_0_arm64.whl.metadata (7.4 kB)
  Using cached grpcio-1.80.0-cp311-cp311-macosx_11_0_universal2.whl.metadata (3.8 kB)
  Using cached keras-3.14.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached charset_normalizer-3.4.7-cp311-cp311-macosx_10_9_universal2.whl.metadata (40 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.me

In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
import requests
import pickle

# 1. Dataset Collection
print("Downloading dataset...")
# Using Alice's Adventures in Wonderland as an example dataset
url = "https://www.gutenberg.org/files/11/11-0.txt"
response = requests.get(url)
text_data = response.text

# 2. Data Preprocessing
# Keep only a subset of text for faster training in this lab (e.g., first 50,000 characters)
text_data = text_data[2000:52000].lower().replace('\n', ' ').replace('\r', '')

# Initialize Tokenizer
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text_data])
total_words = len(tokenizer.word_index) + 1

# Generate Input Sequences
input_sequences = []
# Create sequences of tokens
token_list = tokenizer.texts_to_sequences([text_data])[0]
for i in range(1, len(token_list)):
    n_gram_sequence = token_list[:i+1]
    input_sequences.append(n_gram_sequence)

# Pad Sequences to ensure uniform length
max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

# Prepare Input (X) and Output (y) pairs
# X is all columns except the last one, y is the last column
X, labels = input_sequences[:,:-1], input_sequences[:,-1]
# Convert y to one-hot encoding
y = tf.keras.utils.to_categorical(labels, num_classes=total_words)

print(f"Total Words: {total_words}")
print(f"Max Sequence Length: {max_sequence_len}")
print(f"X shape: {X.shape}, y shape: {y.shape}")


Total Words: 1669
Max Sequence Length: 9651
X shape: (9650, 9650), y shape: (9650, 1669)


Step 2: Model Development & Training

In [3]:
# 3. Model Development
model = Sequential()
# Embedding layer maps words to dense vectors
model.add(Embedding(total_words, 100, input_length=max_sequence_len-1))
# LSTM layer to learn sequences
model.add(LSTM(150, return_sequences=False))
# Dense output layer with softmax for predicting probabilities of the next word
model.add(Dense(total_words, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

# Train the model (Using 20 epochs for demonstration. Increase for better accuracy)
print("Training Model...")
history = model.fit(X, y, epochs=20, verbose=1)

# 4. Save Trained Model and Tokenizer
# We need both the model and the tokenizer for deployment!
model.save('lstm_next_word_model.h5')

with open('tokenizer.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

# Save the max sequence length as well
with open('max_seq_len.txt', 'w') as f:
    f.write(str(max_sequence_len))

print("Model and Tokenizer saved successfully!")


/Users/kirannandi/Projects/MINI_PROJECT/.venv/lib/python3.11/site-packages/keras/src/layers/core/embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Training Model...
Epoch 1/20
302/302 ━━━━━━━━━━━━━━━━━━━━ 1934s 6s/step - accuracy: 0.0423 - loss: 6.2738
Epoch 2/20
302/302 ━━━━━━━━━━━━━━━━━━━━ 9751s 32s/step - accuracy: 0.0548 - loss: 5.8545
Epoch 3/20
302/302 ━━━━━━━━━━━━━━━━━━━━ 1943s 6s/step - accuracy: 0.0689 - loss: 5.6487
Epoch 4/20
302/302 ━━━━━━━━━━━━━━━━━━━━ 1426s 5s/step - accuracy: 0.0964 - loss: 5.3879
Epoch 5/20
302/302 ━━━━━━━━━━━━━━━━━━━━ 1461s 5s/step - accuracy: 0.1220 - loss: 5.0988
Epoch 6/20
302/302 ━━━━━━━━━━━━━━━━━━━━ 1910s 6s/step - accuracy: 0.1461 - loss: 4.8137
Epoch 7/20
302/302 ━━━━━━━━━━━━━━━━━━━━ 1699s 6s/step - accuracy: 0.1645 - loss: 4.5548
Epoch 8/20
302/302 ━━━━━━━━━━━━━━━━━━━━ 1841s 6s/step - accuracy: 0.1788 - loss: 4.3129
Epoch 9/20
302/302 ━━━━━━━━━━━━━━━━━━━━ 4518s 15s/step - accuracy: 0.2020 - loss: 4.0821
Epoch 10/20
302/302 ━━━━━━━━━━━━━━━━━━━━ 2568s 9s/step - accuracy: 0.2190 - loss: 3.8637
Epoch 11/20
302/302 ━━━━━━━━━━━━━━━━━━━━ 1919s 6s/step - accuracy: 0.2415 - loss: 3.6455
Epoch 12/2

Model and Tokenizer saved successfully!
